# Companies / Client Data — Cleaning & Quality Analysis

**Source:** `backend/database/data/companies_import.csv` (801 rows) — the customer list the
quote generator auto-fills company & contact details from.

This notebook does an **end-to-end, heavy-duty audit and clean** of that data. It is meant to
be read top-to-bottom: every stage prints what it found and what it changed, so you can judge
each decision yourself before anything touches the database.

### What it produces (into `cleaning_output/`)
| File | Meaning |
|---|---|
| `companies_clean.csv` | **Load-ready.** Rows that were already clean, plus rows fixed by *safe, deterministic* rules. Same 5-column shape as the source, so `php artisan companies:import` consumes it unchanged. |
| `companies_review.csv` | **Quarantine.** Rows with a risky/ambiguous issue (duplicate company, bare franchise name, person-as-company, no contact method, …). Each row carries the **reason(s)** and, where possible, a **suggested fix**. Nothing here is dropped — you decide. |
| `company_namemap.csv` | Suggested `variant → canonical` company-name merges, with the evidence (shared corporate email domain). Review + apply the ones you agree with. |

### Design principles
1. **Safe auto-fix, quarantine the rest.** Only deterministic corrections (trim, phone
   formatting, address parsing, email recovery) are applied automatically. Anything requiring
   judgment (merging two spellings, deleting junk) is *flagged for you*, never done silently.
2. **No row is lost.** `clean + review == source row count`, always (asserted at the end).
3. **Franchise-aware.** `Signarama` / `FASTSIGNS` are franchises: one name spans dozens of
   independent locations. We never merge those on name alone — see §5.

In [1]:
import pandas as pd, numpy as np, re, unicodedata
from collections import Counter, defaultdict
from pathlib import Path
pd.set_option("display.max_colwidth", 46)
pd.set_option("display.width", 160)

# Robust paths: works whether the notebook runs from its own dir or the repo root.
_CAND = [Path("companies_import.csv"), Path("backend/database/data/companies_import.csv")]
SRC = next((p for p in _CAND if p.exists()), _CAND[0])
OUT = SRC.parent / "cleaning_output"
OUT.mkdir(exist_ok=True)
print("source :", SRC.resolve())
print("output :", OUT.resolve())

source : F:\qoute_generator\epic-quote-v3\backend\database\data\companies_import.csv
output : F:\qoute_generator\epic-quote-v3\backend\database\data\cleaning_output


## §1 · Load

The file is **mixed-encoding**: mostly UTF-8 but with stray Windows-1252 bytes (e.g. the
bullet `•` = byte `0x95`) sitting next to real UTF-8 emoji in the address column. A plain
UTF-8 read *crashes* on those bytes. We read with `encoding_errors="replace"` so bad bytes
become the U+FFFD replacement char (which the cleaner then strips) instead of aborting.

In [2]:
raw = pd.read_csv(SRC, dtype=str, encoding="utf-8", encoding_errors="replace").fillna("")
raw.columns = [c.strip() for c in raw.columns]
df = raw.copy()
print(f"{len(df)} rows x {len(df.columns)} cols: {list(df.columns)}")
df.head(6)

801 rows x 5 cols: ['company', 'contact name', 'phone', 'email', 'address']


,company,contact name,phone,email,address
0,Signarama Sarasota,John Mark Attaway,,mark@signarama-sarasota.com,"5355 McIntosh Rd Suite A, Sarasota, FL 342..."
1,Signarama New Market,Nestor Satumba,289.923.2156,nestor@signarama-newmarket.com,"38 Parkside Dr Unit 2, Newmarket, ON L3Y 8..."
2,Onusiv,Chaz Faulhaber,,chaz@onusiv.com,
3,SIGNARAMA NORRISTOWN,Ethan Polto,(215) 459-2300,ethan@signarama-norristown.com,Ethan Polto <ethan@signarama-norristown.co...
4,Signarama Ankeny,Kyle,515.216.1240 (Business),sales@signarama-ankeny.com,"6990 NE 14th St., Ankeny, IA 50023, Iowa T..."
5,Signaramaccphila,Linzy,(215) 238-9050,cs@signaramaccphila.com,


## §2 · Issue census — how bad is it, exactly?

Before writing a single fix, quantify every problem class. These numbers are the justification
for each cleaning rule that follows.

In [3]:
C, CN, PH, EM, AD = "company", "contact name", "phone", "email", "address"
EMAIL_RE = re.compile(r'[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}')

# --- completeness ---
comp = pd.DataFrame({
    "blank": [(df[c].str.strip() == "").sum() for c in df.columns],
}, index=df.columns)
comp["blank %"] = (comp["blank"] / len(df) * 100).round(1)
print("COMPLETENESS"); display(comp)

# --- duplicate company names (case-insensitive) ---
low = df[C].str.strip().str.lower()
vc = low.value_counts()
dupn = vc[vc > 1]
print(f"\nDUPLICATE COMPANY NAMES: {len(dupn)} names span {int(dupn.sum())} rows "
      f"({dupn.sum()/len(df)*100:.0f}% of file)")
display(dupn.head(10).rename("rows").to_frame())

# --- duplicate emails ---
eml = df[EM].str.strip().str.lower()
ev = eml[eml != ""].value_counts()
print(f"\nDUPLICATE EMAILS: {int((ev>1).sum())} emails appear more than once")
display(ev[ev > 1].head(8).rename("rows").to_frame())

# --- phone format chaos ---
shapes = Counter(re.sub(r'\d', '9', p.strip()) for p in df[PH] if p.strip())
print("\nPHONE FORMATS (a digit = 9):")
display(pd.Series(dict(shapes.most_common(12)), name="count").to_frame())

# --- address column pollution ---
pol = pd.Series({
    "contains an email":      df[AD].str.contains("@", regex=False).sum(),
    "contains '<...>' frag":  df[AD].str.contains("<", regex=False).sum(),
    "contains phone digits":  df[AD].str.contains(r'\d{3}[.\-)]\d{3}', regex=True).sum(),
    "notes (sold/for/attn)":  df[AD].str.contains(r'sold|this is for|for:|attn', case=False, regex=True).sum(),
    "dashes-only / empty-ish": df[AD].str.match(r'^\s*-{2,}\s*$').sum(),
}, name="rows")
print("\nADDRESS COLUMN POLLUTION:"); display(pol.to_frame())

COMPLETENESS


,blank,blank %
company,0,0.0
contact name,0,0.0
phone,495,61.8
email,29,3.6
address,50,6.2



DUPLICATE COMPANY NAMES: 106 names span 349 rows (44% of file)


,rows
company,
signarama,22
fastsigns,21
mustangsigns,8
signarama philadelphia (center city),7
mountain dog sign co.,7
mtdogco,7
nh signs,7
signarama downtown,6
signarama redmond,6



DUPLICATE EMAILS: 163 emails appear more than once


,rows
email,
dan@neonmfg.com,8
clay.curtis@signarama.com,6
info@beartoothsigns.com,5
becky@mtdogco.com,5
graphixlabsf@gmail.com,4
dave@signaramaccphila.com,4
fastsigns.154@fastsigns.com,4
jacob@yakimasigncraft.com,4



PHONE FORMATS (a digit = 9):


,count
999-999-9999,102
999.999.9999,50
(999) 999-9999,46
+9 999-999-9999,15
999 999 9999,9
+99999999999,9
+9 (999) 999-9999,6
(999)999-9999,5
9999999999,4
999.999.9999 (Business),3



ADDRESS COLUMN POLLUTION:


,rows
contains an email,219
contains '<...>' frag,158
contains phone digits,240
notes (sold/for/attn),38
dashes-only / empty-ish,1


**Reading the census**

- **Phone** is ~62% blank and, when present, comes in a dozen incompatible shapes.
- **Company names** are massively duplicated — but that's two *different* problems tangled
  together: (a) the same business typed several ways (`Mtdogco` vs `Mountain Dog Sign Co.`),
  and (b) franchise roots (`Signarama`) that are genuinely many businesses. §5 separates them.
- The **address column is a dumpster** — it has absorbed emails, phone numbers, `<name@x>`
  fragments and freeform notes. Most of the "email" and "phone" signal in this file is
  actually hiding in `address`, which is why the cleaner *mines* it rather than trusting the
  dedicated columns.

## §3 · Cleaning primitives (safe, deterministic transforms)

Each function does exactly one narrow, defensible thing and is **idempotent** (running it
twice changes nothing). They never guess — anything ambiguous is left for the classifier in §6.
The self-tests at the bottom are the contract; if they pass, the transforms behave as described.

In [4]:
# ---- text normalization -----------------------------------------------------
def norm_ws(s):
    """NFKC (folds nbsp->space), drop zero-width/BOM, turn U+FFFD mojibake into a
    space, then collapse whitespace. Pure-codepoint so the source stays ASCII."""
    s = unicodedata.normalize("NFKC", str(s))
    s = "".join("" if ord(ch) in (0x200B, 0xFEFF)
                else (" " if ord(ch) == 0xFFFD else ch) for ch in s)
    return re.sub(r"\s+", " ", s).strip()

def strip_emoji(s):
    return re.sub(r'[\U0001F000-\U0001FAFF\U00002600-\U000027BF\U0000FE0F'
                  r'\U00002190-\U000021FF\U00002B00-\U00002BFF]', '', s)

def valid_email(e):  return bool(e) and bool(re.fullmatch(EMAIL_RE, e))
def domain_of(e):    return e.split("@", 1)[1].lower() if "@" in e else ""
def name_slug(s):    return re.sub(r'[^a-z0-9]', '', norm_ws(s).lower())

FREE_DOMAINS = {"gmail.com","yahoo.com","hotmail.com","outlook.com","aol.com","icloud.com",
                "comcast.net","me.com","live.com","msn.com","att.net","verizon.net",
                "sbcglobal.net","ymail.com","protonmail.com"}
# umbrella franchise domains: ONE domain, MANY independent locations -> never merge on domain
UMBRELLA_DOMAINS = {"signarama.com","fastsigns.com","speedpro.com","image360.com",
                    "signgypsies.com","alliancefranchise.com"}
FRANCHISE_ROOTS = {"signarama","fast signs","fastsigns","speedpro","image360",
                   "sign gypsies","signgypsies"}
ROLE_RE = re.compile(r'^(office|office manager|office administrator|owner|manager|sales|'
    r'sales ?rep(resentative)?|sales & marketing|project manager|accounting|accounts?|'
    r'front desk|admin|administrator|receptionist|estimator|designer|purchasing|info|'
    r'billing|customer service|n/?a|none|unknown|test|tbd)'
    r'([ /|&\-]+(office|manager|owner|sales|project manager|admin|marketing))*$', re.I)

# words that mark a string as a COMPANY (vs a person's name) -> used to pick canonicals
COMPANY_KW = re.compile(r'\b(signs?|signarama|fastsigns|graphics?|print(ing|s)?|media|'
    r'imaging|banners?|wraps?|displays?|design|studio|group|co|company|inc|llc|ltd|corp|'
    r'enterprises?|solutions?|services?|works?|shop|store|center|centre|speedpro|'
    r'image360|minuteman)\b', re.I)
def has_company_kw(name):
    return bool(COMPANY_KW.search(name)) or name_slug(name)[:8] in {"signaram", "fastsign"}
def is_franchise_root(name):
    return name_slug(name) in {name_slug(x) for x in FRANCHISE_ROOTS}

# ---- phone -----------------------------------------------------------------
def clean_phone(rawp):
    """-> (normalized, email_found, note). US -> (NPA) NXX-XXXX, keeps ext + a
    (business/office/...) label; intl -> +digits; unparseable kept verbatim."""
    s = norm_ws(rawp)
    if not s: return "", "", ""
    notes = []
    m = EMAIL_RE.search(s); email_in = m.group(0).lower() if m else ""
    if email_in:
        s = norm_ws(s.replace(m.group(0), "")); notes.append("email_in_phone")
    ext = ""
    em = re.search(r'(?:ext|x|extension)\.?\s*(\d{1,6})', s, re.I)
    if em: ext = em.group(1); s = s[:em.start()] + s[em.end():]
    lbl = ""
    lm = re.search(r'\(?([A-Za-z]*(business|office|cell|mobile|fax|home|direct)[A-Za-z]*)\)?', s, re.I)
    if lm: lbl = lm.group(2).lower(); s = s[:lm.start()] + " " + s[lm.end():]
    s = re.sub(r'^[A-Za-z]\s*:\s*', '', s.strip())
    digits = re.sub(r'\D', '', s)
    intl = norm_ws(rawp).startswith("+") or len(digits) > 11
    if not intl and len(digits) == 10:
        out = f"({digits[0:3]}) {digits[3:6]}-{digits[6:]}"
    elif not intl and len(digits) == 11 and digits[0] == "1":
        d = digits[1:]; out = f"({d[0:3]}) {d[3:6]}-{d[6:]}"
    elif intl and 8 <= len(digits) <= 15:
        out = "+" + digits
    else:
        out = norm_ws(rawp)
        if not re.fullmatch(r'[\d\s().+\-/#xX]+', norm_ws(rawp).replace("ext", "")):
            notes.append("phone_unparseable")
    if ext: out = f"{out} x{ext}"
    if lbl: out = f"{out} ({lbl})"
    return out, email_in, ";".join(notes)

# ---- junk detectors (a "name" that is really a phone / URL / label) --------
URL_RE   = re.compile(r'^(https?://|www\.)|\.(com|net|org|biz|co|us|ca)\b', re.I)
PHONEY_RE = re.compile(r'^[\d\s().+\-/#]{7,}$')
LABEL_RE = re.compile(r'^(full name|name|phone number|phone|email address|email|web|website|contact)\b[\s:]*', re.I)

def is_phone_like(s):
    """True when the whole string is phone digits/punctuation, incl. 'x cell: y' pairs."""
    t = re.sub(r'\b(cell|office|direct|fax|tel|ph|mobile|main)\b\.?\s*:?', ' ', s, flags=re.I)
    t = norm_ws(re.sub(r'[|/,]', ' ', t))
    return bool(t) and bool(PHONEY_RE.match(t)) and len(re.sub(r'\D', '', t)) >= 7

def is_url_like(s):
    return bool(URL_RE.search(norm_ws(s))) and "@" not in s

# ---- contact name ----------------------------------------------------------
def clean_contact(rawn):
    """Strip '<email>' / 'Email: x' fragments and leading form labels, then reject a
    'name' that is really a phone number or a website. -> (name, email, phone, junk)."""
    s = norm_ws(rawn)
    m = EMAIL_RE.search(s); extracted = m.group(0).lower() if m else ""
    s = re.sub(r'<[^>]*>?', '', s)
    s = re.sub(r'\bemail\s*:?\s*\S+@\S+', '', s, flags=re.I)
    s = re.sub(EMAIL_RE, '', s)
    s = norm_ws(s.strip(" ,;|<>"))
    s = norm_ws(LABEL_RE.sub('', s))          # "Full Name Signarama" -> "Signarama"
    phone_out, junk = "", ""
    if s and is_phone_like(s):                # the name IS a phone -> move it, drop the name
        phone_out, s, junk = s, "", "contact_was_phone"
    elif s and is_url_like(s):                # the name IS a website -> drop it
        s, junk = "", "contact_was_url"
    return s, extracted, phone_out, junk

# ---- address ---------------------------------------------------------------
CITY_STATE_RE = re.compile(r',\s*([A-Za-z][A-Za-z .\'\-]+?),?\s+([A-Z]{2})\.?\s+'
                           r'(\d{5}(?:-\d{4})?|[A-Za-z]\d[A-Za-z]\s?\d[A-Za-z]\d)')
NOTE_TAIL_RE  = re.compile(r'\b(this is (?:sold|for)|for|attn|re)\s*:?.*$', re.I)
PHONE_ONLY_RE = re.compile(r'^[\d\s().+\-/#]{7,}$')

def clean_address(rawa):
    """-> (address, emails_found, phone_only, city, state, zip, note_removed)."""
    s = norm_ws(rawa)
    if not s or set(s) <= set("-–—_ "):
        return "", [], "", "", "", "", ""
    emails = [e.lower() for e in EMAIL_RE.findall(s)]
    for e in EMAIL_RE.findall(s): s = s.replace(e, "")
    s = re.sub(r'<[^>]*>?', '', s)
    note = ""
    nm = NOTE_TAIL_RE.search(s)
    if nm: note = norm_ws(nm.group(0)); s = s[:nm.start()]
    s = strip_emoji(s)
    # web-form leakage: an IP address, an IANA timezone, a bare website, and "Phone Number:"/
    # "Email Address:" labels all rode in from a scraped contact form. None are an address.
    s = re.sub(r'\b\d{1,3}(?:\.\d{1,3}){3}\b', '', s)                    # 99.162.252.90
    s = re.sub(r'\b[A-Za-z]+/[A-Za-z_]+\b', '', s)                       # America/New_York
    s = re.sub(r'\b(?:https?://|www\.)\S+', '', s, flags=re.I)           # www.example.com
    s = re.sub(r'\b(phone number|email address|full name|website|web)\b\s*:?', '', s, flags=re.I)
    city = state = zipc = ""
    cm = CITY_STATE_RE.search(s)
    if cm: city, state, zipc = cm.group(1).strip(), cm.group(2), cm.group(3)
    s = re.sub(r'\S*[<>@]\S*', '', s)                 # kill residual email/angle fragments
    s = norm_ws(s).strip(" ,;|`\"'–—/-")
    phone_only = ""
    if s and PHONE_ONLY_RE.match(s):
        phone_only = s; s = ""
    return s, emails, phone_only, city, state, zipc, note

# ---- company name ----------------------------------------------------------
def smart_title(s):
    def fix(tok):
        core = re.sub(r'[^A-Za-z]', '', tok)
        return tok if (len(core) <= 3 and core.isupper()) else tok[:1].upper() + tok[1:].lower()
    return " ".join(fix(t) for t in s.split())

def clean_company(rawc):
    s = norm_ws(rawc).strip(' ,;|`"\'')
    letters = re.sub(r'[^A-Za-z]', '', s)
    cased = False
    if letters and s.upper() == s and len(letters) > 3:     # ALL CAPS -> Title Case
        s = smart_title(s); cased = True
    return s, cased

# ---- self-tests (the contract) --------------------------------------------
assert clean_phone("515.216.1240 (Business)")[0] == "(515) 216-1240 (business)"
assert clean_phone("(215) 459-2300")[0] == "(215) 459-2300"
assert clean_phone("289.923.2156")[0] == "(289) 923-2156"
assert clean_contact("Randy Rasmussen <randy.rasmussen@fastsigns.com")[0] == "Randy Rasmussen"
assert clean_contact("MIke Email: 154@fastsigns.com")[1] == "154@fastsigns.com"
# a "name" that is really a phone -> name dropped, number rescued into the phone slot
assert clean_contact("954-926-3380 cell: 863-356-7374")[0] == ""
assert clean_contact("954-926-3380 cell: 863-356-7374")[3] == "contact_was_phone"
assert clean_contact("954-926-3380 cell: 863-356-7374")[2] != ""
# a "name" that is really a website -> dropped
assert clean_contact("www.signarama-hollywoodfl.com")[0] == ""
assert clean_contact("Web www.signarama-danbury.com")[0] == ""
assert clean_contact("Full Name Signarama Hollywood")[0] == "Signarama Hollywood"
assert not is_url_like("trent ulrey <valleysignssolutions@gmail.com>")   # an email is not a URL
# web-form leakage never survives into an address
_b = clean_address("Hallandale, Florida, United States, America/New_York, 99.162.252.90")
assert "99.162" not in _b[0] and "America/New_York" not in _b[0], _b
assert clean_company("SIGNARAMA NORRISTOWN")[0] == "Signarama Norristown"
_a = clean_address('5355 McIntosh Rd, Sarasota, FL 34233, This is sold!!!, Chelsea <c@signarama.com>')
assert _a[3] == "Sarasota" and _a[4] == "FL" and "@" not in _a[0] and "<" not in _a[0]
assert clean_address("-----")[0] == ""
print("self-tests PASS - primitives behave as documented")

self-tests PASS - primitives behave as documented


## §4 · Stage 1 — apply the safe fixes, row by row

For every row we normalize the company name, de-pollute the contact/phone/address fields, and
**recover** the email from wherever it's actually hiding (email col → contact `<...>` → phone →
address). `fixes` records exactly what changed, so nothing is a black box.

In [5]:
recs = []
for i, r in df.iterrows():
    fixes = []
    co_raw, cn_raw, ph_raw, em_raw, ad_raw = r[C], r[CN], r[PH], r[EM], r[AD]

    company, cap = clean_company(co_raw)
    if cap: fixes.append("company_titlecased")
    contact, cn_email, cn_phone, cn_junk = clean_contact(cn_raw)
    if cn_junk: fixes.append(cn_junk)                       # name was really a phone / website
    elif norm_ws(contact) != norm_ws(cn_raw): fixes.append("contact_stripped")
    ph_out, ph_email, ph_note = clean_phone(ph_raw)
    if not ph_out and cn_phone:                             # rescue the phone hiding in the name
        ph_out = clean_phone(cn_phone)[0]; fixes.append("phone_from_contact")
    if ph_out != norm_ws(ph_raw) and ph_out and "phone_from_contact" not in fixes:
        fixes.append("phone_normalized")

    email = ""
    for cand in (em_raw, cn_email, ph_email):
        m = EMAIL_RE.search(str(cand))
        if m: email = m.group(0).lower(); break
    ad_out, ad_emails, ad_phone_only, city, state, zipc, _ = clean_address(ad_raw)
    if not email and ad_emails:
        email = ad_emails[0]; fixes.append("email_from_address")
    elif email and not EMAIL_RE.search(em_raw or ""):
        fixes.append("email_recovered")
    ok = valid_email(email)
    if not ph_out and ad_phone_only:
        ph_out = clean_phone(ad_phone_only)[0]; fixes.append("phone_from_address")
    if norm_ws(ad_out) != norm_ws(ad_raw): fixes.append("address_cleaned")

    recs.append(dict(idx=i, company=company, contact=contact, phone=ph_out,
        email=email if ok else "", address=ad_out, city=city, state=state, zip=zipc,
        domain=domain_of(email) if ok else "", slug=name_slug(company),
        fixes=";".join(sorted(set(fixes))),
        raw_company=co_raw, raw_contact=cn_raw, raw_phone=ph_raw,
        raw_email=em_raw, raw_address=ad_raw))
clean = pd.DataFrame(recs)

fixcount = Counter(f for s in clean["fixes"] for f in s.split(";") if f)
print("Rows touched per fix:")
display(pd.Series(dict(fixcount.most_common()), name="rows").to_frame())

# a few concrete before/after examples so you can eyeball the transforms
ex = clean[clean["fixes"].str.contains("phone_normalized|address_cleaned|email_from_address")].head(6)
print("\nBEFORE -> AFTER (samples):")
for _, r in ex.iterrows():
    print(f"  [{r['fixes']}]")
    print(f"    phone   {r['raw_phone']!r:30} -> {r['phone']!r}")
    print(f"    address {r['raw_address'][:44]!r:46} -> {r['address'][:40]!r}")

Rows touched per fix:


,rows
address_cleaned,294
phone_normalized,250
contact_stripped,48
company_titlecased,38
phone_from_address,12
email_from_address,10
contact_was_url,10
contact_was_phone,9
phone_from_contact,9



BEFORE -> AFTER (samples):
  [address_cleaned]
    phone   ''                             -> ''
    address '5355 McIntosh Rd Suite A, Sarasota, FL 34233' -> '5355 McIntosh Rd Suite A, Sarasota, FL 3'
  [phone_normalized]
    phone   '289.923.2156'                 -> '(289) 923-2156'
    address '38 Parkside Dr Unit 2, Newmarket, ON L3Y 8J9' -> '38 Parkside Dr Unit 2, Newmarket, ON L3Y'
  [address_cleaned;company_titlecased]
    phone   '(215) 459-2300'               -> '(215) 459-2300'
    address 'Ethan Polto <ethan@signarama-norristown.com>' -> 'Ethan Polto , 310 W JOHNSON HWY SUITE 10'
  [phone_normalized]
    phone   '515.216.1240 (Business)'      -> '(515) 216-1240 (business)'
    address '6990 NE 14th St., Ankeny, IA 50023, Iowa Tar' -> '6990 NE 14th St., Ankeny, IA 50023, Iowa'
  [phone_normalized]
    phone   '856-829-1460'                 -> '(856) 829-1460'
    address '707 West Spring Garden Street, Palmyra, NJ 0' -> '707 West Spring Garden Street, Palmyra, '
  [phone_norm

## §5 · Stage 2 — find duplicate companies (the careful part)

Two rows are the **same company** when they share a **corporate email domain** that is not a
free provider (gmail) and not a franchise umbrella (`signarama.com`). Example:
`aaron@mtdogco.com` appears as both `Mountain Dog Sign Co.` and `Mtdogco` — same business,
two spellings.

**We do not rewrite names here.** Auto-merging is dangerous: `Signarama Fort Washington` and
`Signarama Norristown` can share a franchisee's sub-domain yet be distinct locations. Instead
we (a) emit a **suggested** `variant → canonical` map for you to approve, and (b) flag every
row in a multi-spelling cluster for review. The canonical is chosen to be a *company-looking*
name (never a person's name that happened to land in the company column).

In [6]:
dom_names = defaultdict(Counter)
for _, r in clean.iterrows():
    d = r["domain"]
    if d and d not in FREE_DOMAINS and d not in UMBRELLA_DOMAINS and r["company"]:
        dom_names[d][r["company"]] += 1

dup_domains, namemap_rows = {}, []
for d, ctr in dom_names.items():
    if len(ctr) < 2:                       # one spelling on this domain -> not a duplicate
        continue
    label = d.split(".")[0].replace("-", "")
    def is_cand(n):                        # candidate canonical must look like the company
        s = name_slug(n)
        return has_company_kw(n) or s.startswith(label) or label.startswith(s[:8])
    cands = [n for n in ctr if is_cand(n)] or list(ctr)
    best = sorted(cands, key=lambda n: (ctr[n], len(n)), reverse=True)[0]
    dup_domains[d] = best
    for v in ctr:
        namemap_rows.append(dict(domain=d, variant=v, suggested_canonical=best,
                                 count=ctr[v], is_suggested=int(v != best)))

clean["dup_domain"] = clean["domain"].apply(lambda d: d in dup_domains)
clean["suggested_company"] = clean.apply(
    lambda r: dup_domains.get(r["domain"], "")
    if r["dup_domain"] and r["company"] != dup_domains.get(r["domain"], "") else "", axis=1)

print(f"corporate domains carrying >1 name spelling : {len(dup_domains)}")
print(f"rows sitting in a duplicate cluster          : {int(clean['dup_domain'].sum())}")
print(f"rows whose name differs from the suggestion  : {int((clean['suggested_company']!='').sum())}")
namemap = pd.DataFrame(namemap_rows)
print("\nSuggested merges (sample):")
display(namemap[namemap["is_suggested"] == 1]
        [["variant", "suggested_canonical", "domain", "count"]].head(12))

corporate domains carrying >1 name spelling : 92
rows sitting in a duplicate cluster          : 369
rows whose name differs from the suggestion  : 202

Suggested merges (sample):


,variant,suggested_canonical,domain,count
1,Madison Lojeski | Sales Manager,Signarama Norristown,signarama-norristown.com,1
2,Signarama Fort Washington,Signarama Norristown,signarama-norristown.com,2
3,Office Manager,Signarama Norristown,signarama-norristown.com,1
4,Signarama NorristownEthan Polto,Signarama Norristown,signarama-norristown.com,1
5,"Signarama Fort Washington, PA",Signarama Norristown,signarama-norristown.com,1
7,"Kyle Meadows, Account Executive",Signarama Ankeny,signarama-ankeny.com,1
8,Signaramaccphila,Signarama Philadelphia (Center City),signaramaccphila.com,2
10,"Signarama Philadelphia (Center City), PA",Signarama Philadelphia (Center City),signaramaccphila.com,1
11,Signarama Center City,Signarama Philadelphia (Center City),signaramaccphila.com,1
12,Signarama Philadelphia,Signarama Philadelphia (Center City),signaramaccphila.com,1


## §6 · Stage 3 — classify every row: `clean` / `fixed` / `review`

- **clean** — nothing was wrong, nothing changed.
- **fixed** — only safe deterministic fixes applied → still load-ready.
- **review** — at least one issue needs a human. Reasons:

| reason | meaning |
|---|---|
| `possible_duplicate_company` | shares a corporate domain with another spelling (see suggestion) |
| `bare_franchise_name` | name is just `Signarama`/`Fastsigns` with no location — identity unknown |
| `company_name_is_person` | company field holds a person's name (== contact, no company word) |
| `role_or_junk_company_name` | company is a job title (`Owner`, `Office Manager`) |
| `company_looks_like_address` | column-shifted row — an address landed in the company field |
| `email_shared_across_companies` | same email on different companies (free/umbrella domain) |
| `no_contact_method` | neither a phone nor a valid email — not actionable |
| `email_invalid` | an email was present but malformed |

In [7]:
email_companies = defaultdict(set)
for _, r in clean.iterrows():
    if r["email"]: email_companies[r["email"]].add(r["slug"])
cross_email = {e for e, s in email_companies.items() if len(s) > 1}

CO_LOOKS_LIKE_ADDR = re.compile(
    r'\b[A-Z]{2}\s+\d{5}\b|\bunited states\b|\bcanada\b|'
    r'\d{1,5}\s+\w+\s+(st|street|ave|avenue|rd|road|blvd|dr|drive|hwy|suite|unit)\b', re.I)

def review_reasons(r):
    out = []
    if is_franchise_root(r["company"]):                 out.append("bare_franchise_name")
    if ROLE_RE.match(norm_ws(r["company"])):            out.append("role_or_junk_company_name")
    if len(name_slug(r["company"])) < 2:                out.append("company_name_too_short")
    if CO_LOOKS_LIKE_ADDR.search(r["company"]):         out.append("company_looks_like_address")
    if (name_slug(r["company"]) == name_slug(r["contact"]) and r["contact"]
            and not has_company_kw(r["company"])):      out.append("company_name_is_person")
    if not r["contact"]:                                out.append("contact_name_missing")
    # "Francisco Rodriguez Villegas | Signarama Hollywood" — a person AND a company in one
    # cell. Splitting is a judgement call (which side is the company?), so a human decides.
    if "|" in r["raw_company"]:                         out.append("company_has_person_prefix")
    if not r["phone"] and not r["email"]:               out.append("no_contact_method")
    if r["raw_email"].strip() and not r["email"]:       out.append("email_invalid")
    if r["suggested_company"]:                          out.append("possible_duplicate_company")
    if (r["email"] in cross_email and "possible_duplicate_company" not in out
            and (not r["domain"] or r["domain"] in FREE_DOMAINS
                 or r["domain"] in UMBRELLA_DOMAINS)):  out.append("email_shared_across_companies")
    return out

clean["review"] = clean.apply(lambda r: ";".join(review_reasons(r)), axis=1)
clean["status"] = np.where(clean["review"] != "", "review",
                    np.where(clean["fixes"] != "", "fixed", "clean"))

print("STATUS:"); display(clean["status"].value_counts().rename("rows").to_frame())
rc = Counter(x for s in clean["review"] if s for x in s.split(";"))
print("REVIEW REASONS:")
display(pd.Series(dict(rc.most_common()), name="rows").to_frame())

# eyeball a few of each reason
for reason in ["possible_duplicate_company", "bare_franchise_name",
               "company_name_is_person", "no_contact_method", "company_looks_like_address"]:
    sub = clean[clean["review"].str.contains(reason)]
    print(f"\n--- {reason}  ({len(sub)} rows) ---")
    display(sub[["company", "suggested_company", "contact", "email"]].head(3))

STATUS:


,rows
status,
review,371
fixed,262
clean,168


REVIEW REASONS:


,rows
possible_duplicate_company,202
email_shared_across_companies,121
bare_franchise_name,44
company_name_is_person,28
contact_name_missing,25
company_has_person_prefix,10
no_contact_method,5
role_or_junk_company_name,5
company_looks_like_address,1



--- possible_duplicate_company  (202 rows) ---


...

,company,suggested_company,contact,email
8,Prismsignstx,,Prism Signs TX,info@prismsignstx.com
48,Taher Muhammad,Prismsignstx,Taher Muhammad,info@prismsignstx.com
87,Rod Muffett,,Rod Muffett,



--- no_contact_method  (5 rows) ---


,company,suggested_company,contact,email
87,Rod Muffett,,Rod Muffett,
98,Fast Signs Lynnwood,,Fast Signs,
179,"Signarama Sioux Falls, SD",,Hassan Qureshi,



--- company_looks_like_address  (1 rows) ---


,company,suggested_company,contact,email
277,Sarasota FL 34238 United States,Neon Mfg,Dan Defilla,dan@neonmfg.com


## §7 · Write the outputs

`companies_clean.csv` and `companies_review.csv` use the **exact 5-column schema** the existing
importer expects (`company, contact name, phone, email, address`), so the clean file drops
straight into `php artisan companies:import`.

In [8]:
def to5(sub):
    return pd.DataFrame({"company": sub["company"], "contact name": sub["contact"],
                         "phone": sub["phone"], "email": sub["email"], "address": sub["address"]})

clean_rows  = clean[clean["status"] != "review"]
review_rows = clean[clean["status"] == "review"]

to5(clean_rows).to_csv(OUT / "companies_clean.csv", index=False)

rev = to5(review_rows).copy()
rev.insert(0, "review_reasons", review_rows["review"].values)
rev.insert(1, "orig_row", review_rows["idx"].values + 2)          # +2 = header + 1-indexing
rev.insert(2, "suggested_company", review_rows["suggested_company"].values)
rev.to_csv(OUT / "companies_review.csv", index=False)

namemap.to_csv(OUT / "company_namemap.csv", index=False)

summary = pd.DataFrame({
    "rows": [len(clean_rows), len(review_rows), len(namemap[namemap.is_suggested == 1])],
}, index=["companies_clean.csv (load-ready)",
          "companies_review.csv (needs you)",
          "company_namemap.csv (suggested merges)"])
print("WROTE ->", OUT)
display(summary)

WROTE -> cleaning_output


,rows
companies_clean.csv (load-ready),430
companies_review.csv (needs you),371
company_namemap.csv (suggested merges),174


## §8 · Validation — prove the clean file is actually clean

Hard invariants on the load-ready output. If any of these is non-zero, the pipeline has a bug —
do **not** import.

In [9]:
c5 = pd.read_csv(OUT / "companies_clean.csv", dtype=str).fillna("")
checks = {
    "invalid emails":                 sum(bool(e) and not valid_email(e) for e in c5["email"]),
    "addresses still polluted (< @)": int(c5["address"].str.contains(r'[<@]', regex=True).sum()),
    "bare franchise names":           sum(is_franchise_root(x) for x in c5["company"]),
    "blank company names":            int((c5["company"].str.strip() == "").sum()),
    "U+FFFD mojibake left":           int(c5.apply(lambda r: any(chr(0xFFFD) in str(v) for v in r), axis=1).sum()),
    "rows lost (clean+review vs src)":(len(clean_rows) + len(review_rows)) - len(df),
}
res = pd.Series(checks, name="count").to_frame()
res["ok"] = res["count"] == 0
display(res)
assert res["ok"].all(), "INVARIANT FAILED - inspect before importing"
print("ALL INVARIANTS GREEN - clean file is safe to import; no rows lost.")
print(f"distinct companies in clean file: {c5['company'].str.lower().nunique()}")

,count,ok
invalid emails,0,True
addresses still polluted (< @),0,True
bare franchise names,0,True
blank company names,0,True
U+FFFD mojibake left,0,True
rows lost (clean+review vs src),0,True


ALL INVARIANTS GREEN - clean file is safe to import; no rows lost.
distinct companies in clean file: 311


## §9 · Your review workflow → load into the DB

1. **Open `cleaning_output/companies_review.csv`.** Work top-down by `review_reasons`:
   - `possible_duplicate_company` → accept the `suggested_company` (edit the `company` cell to
     match it) or reject it. Approved merges can also be captured in `company_namemap.csv`.
   - `bare_franchise_name` → rename to the real location (hint lives in the email localpart,
     e.g. `stonemtnga.sales@` ⇒ *Signarama Stone Mountain GA*), or leave to collapse under the
     generic franchise entry.
   - `company_name_is_person` / `role_or_junk_company_name` → put the real company in `company`,
     the person stays in `contact name`.
   - `company_looks_like_address` → fix the column shift by hand (rare — a couple of rows).
   - `no_contact_method` → decide whether the row is worth keeping at all.
2. **Move the rows you've fixed** out of `companies_review.csv` and append them to
   `companies_clean.csv` (same columns).
3. **Dry-run the import** (writes nothing), then run it for real:
   ```bash
   cd backend
   php artisan companies:import database/data/cleaning_output/companies_clean.csv --dry-run
   php artisan companies:import database/data/cleaning_output/companies_clean.csv
   ```
   The importer dedupes companies by name and contacts by (company, name), so re-running is
   safe and rows sharing one company name become one company with multiple contacts.

The optional helper below **applies an approved `company_namemap.csv`** to a chosen input file
(set `apply=True` after you've pruned the map to only merges you agree with).

In [10]:
def apply_namemap(in_csv, out_csv, mapping_csv=OUT / "company_namemap.csv"):
    src = pd.read_csv(in_csv, dtype=str).fillna("")
    mp = pd.read_csv(mapping_csv, dtype=str).fillna("")
    mp = mp[mp["is_suggested"] == "1"]
    lut = {name_slug(v): c for v, c in zip(mp["variant"], mp["suggested_canonical"])}
    n = 0
    def fix(name):
        nonlocal n
        c = lut.get(name_slug(name))
        if c and c != name: n += 1; return c
        return name
    src["company"] = src["company"].map(fix)
    src.to_csv(out_csv, index=False)
    print(f"applied {n} name merges -> {out_csv}")

# apply = False  # flip to True AFTER pruning company_namemap.csv to merges you approve
apply = False
if apply:
    apply_namemap(OUT / "companies_clean.csv", OUT / "companies_clean_merged.csv")
else:
    print("Review company_namemap.csv, prune to approved merges, then set apply=True and re-run.")

Review company_namemap.csv, prune to approved merges, then set apply=True and re-run.
